In [8]:
!pip install --upgrade websockets urllib3 PyYAML

In [10]:
!pip install ccxt yfinance alpaca-py pandas requests python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 2.2 MB/s eta 0:00:00


In [11]:
import ccxt, yfinance, pandas, requests
from alpaca.trading.client import TradingClient

print("ccxt:", ccxt.__version__)
print("yfinance:", yfinance.__version__)
print("pandas:", pandas.__version__)
print("alpaca-py: OK")
print("\nAll good — proceed to Cell 2")

ccxt: 4.5.51
yfinance: 0.2.66
pandas: 2.2.2
alpaca-py: OK

All good — proceed to Cell 2


In [12]:

# =============================================================================
# CELL 2 — Python Fundamentals You Need (Read + Run This)
# If you're new to Python, this cell teaches you the exact syntax
# used throughout this project. Don't skip it.
# =============================================================================

# --- VARIABLES ---
name = "TradingAgent"           # String
capital = 20.0                  # Float (decimal number)
is_live = False                 # Boolean (True/False — capital T and F matter in Python)
markets = ["crypto", "us", "india", "asx"]  # List (like an array in JS)

print(f"Agent: {name} | Capital: ${capital} | Live: {is_live}")
# f-strings = template literals in JS — f"hello {variable}"

# --- DICTIONARIES (like JS objects) ---
trade_signal = {
    "asset": "BTC/USDT",
    "action": "BUY",
    "confidence": 0.85,
    "reason": "RSI oversold + bullish news"
}
print(trade_signal["asset"])   # Access by key
print(trade_signal.get("action", "HOLD"))  # Safe access with default

# --- FUNCTIONS ---
def calculate_position_size(capital, risk_percent):
    """
    How much money to put into one trade.
    Rule: Never risk more than X% of capital on a single trade.
    """
    return capital * (risk_percent / 100)

position = calculate_position_size(capital=20.0, risk_percent=10)
print(f"Max position size: ${position:.2f}")  # :.2f = 2 decimal places

# --- LOOPS ---
for market in markets:
    print(f"  -> Will trade on: {market.upper()}")

# --- CONDITIONALS ---
rsi_value = 28  # RSI below 30 = oversold = potential BUY signal
if rsi_value < 30:
    print("RSI: OVERSOLD — potential BUY signal")
elif rsi_value > 70:
    print("RSI: OVERBOUGHT — potential SELL signal")
else:
    print("RSI: NEUTRAL — HOLD")

# --- ERROR HANDLING (critical for API calls) ---
try:
    result = 100 / 0  # This will fail
except ZeroDivisionError as e:
    print(f"Error caught: {e} — handling gracefully")

# --- CLASSES (you'll need this for the RL environment later) ---
class TradingAccount:
    def __init__(self, starting_capital):
        self.capital = starting_capital
        self.trades = []

    def buy(self, asset, amount):
        if amount > self.capital:
            print(f"Insufficient capital. Have ${self.capital}, need ${amount}")
            return
        self.capital -= amount
        self.trades.append({"asset": asset, "amount": amount, "type": "BUY"})
        print(f"Bought ${amount} of {asset}. Remaining capital: ${self.capital:.2f}")

    def show_history(self):
        for trade in self.trades:
            print(f"  {trade['type']} ${trade['amount']} of {trade['asset']}")

account = TradingAccount(starting_capital=20.0)
account.buy("BTC/USDT", 10.0)
account.buy("ETH/USDT", 5.0)
account.buy("SOL/USDT", 50.0)  # Should fail — not enough capital
account.show_history()


Agent: TradingAgent | Capital: $20.0 | Live: False
BTC/USDT
BUY
Max position size: $2.00
  -> Will trade on: CRYPTO
  -> Will trade on: US
  -> Will trade on: INDIA
  -> Will trade on: ASX
RSI: OVERSOLD — potential BUY signal
Error caught: division by zero — handling gracefully
Bought $10.0 of BTC/USDT. Remaining capital: $10.00
Bought $5.0 of ETH/USDT. Remaining capital: $5.00
Insufficient capital. Have $5.0, need $50.0
  BUY $10.0 of BTC/USDT
  BUY $5.0 of ETH/USDT


In [14]:
# CELL 3 — FIXED: KuCoin instead of Binance (works in India)
import ccxt
import datetime

# KuCoin works in India, no geo-restriction, no API key for public data
exchange = ccxt.kucoin({
    'enableRateLimit': True,
})

print(f"Connected to: {exchange.name}")

# Fetch live BTC price
ticker = exchange.fetch_ticker('BTC/USDT')

print("\n" + "="*50)
print("BTC/USDT — LIVE MARKET DATA (KuCoin)")
print("="*50)
print(f"Last Price:    ${ticker['last']:,.2f}")
print(f"24h High:      ${ticker['high']:,.2f}")
print(f"24h Low:       ${ticker['low']:,.2f}")
print(f"24h Change:    {ticker['percentage']:.2f}%")
print(f"24h Volume:    {ticker['baseVolume']:,.2f} BTC")
print(f"Timestamp:     {datetime.datetime.utcfromtimestamp(ticker['timestamp']/1000)} UTC")

btc_you_can_buy = 20 / ticker['last']
print(f"\nWith $20 you can buy: {btc_you_can_buy:.8f} BTC")

Connected to: KuCoin

BTC/USDT — LIVE MARKET DATA (KuCoin)
Last Price:    $79,791.20
24h High:      $80,764.60
24h Low:       $78,219.50
24h Change:    1.55%
24h Volume:    5,283.57 BTC
Timestamp:     2026-05-04 23:49:36.831000 UTC

With $20 you can buy: 0.00025065 BTC


/tmp/ipykernel_14394/3632858072.py:23: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  print(f"Timestamp:     {datetime.datetime.utcfromtimestamp(ticker['timestamp']/1000)} UTC")


In [15]:
import ccxt
import pandas as pd  # Pandas = the most important library for data in Python

exchange = ccxt.kucoin({'enableRateLimit': True})

# Assets your agent will eventually trade
watchlist = [
    'BTC/USDT',   # Bitcoin
    'ETH/USDT',   # Ethereum
    'SOL/USDT',   # Solana
    'BNB/USDT',   # BNB
    'ADA/USDT',   # Cardano
]

print("Fetching live prices for watchlist...\n")

market_snapshot = []

for symbol in watchlist:
    try:
        ticker = exchange.fetch_ticker(symbol)
        market_snapshot.append({
            'symbol':      symbol,
            'price':       ticker['last'],
            'change_24h':  round(ticker['percentage'], 2),
            'volume_24h':  round(ticker['quoteVolume'] / 1_000_000, 2),  # In millions
            'high_24h':    ticker['high'],
            'low_24h':     ticker['low'],
        })
        print(f"  {symbol:<12} ${ticker['last']:>12,.4f}   {ticker['percentage']:>+6.2f}%")
    except Exception as e:
        print(f"  {symbol:<12} ERROR: {e}")

# Convert to Pandas DataFrame — this is how you'll work with all data
df = pd.DataFrame(market_snapshot)
print("\n--- DataFrame View ---")
print(df.to_string(index=False))

# Simple analysis using Pandas
print(f"\nBest performer (24h):  {df.loc[df['change_24h'].idxmax(), 'symbol']} ({df['change_24h'].max()}%)")
print(f"Worst performer (24h): {df.loc[df['change_24h'].idxmin(), 'symbol']} ({df['change_24h'].min()}%)")
print(f"Highest volume:        {df.loc[df['volume_24h'].idxmax(), 'symbol']} (${df['volume_24h'].max()}M)")


Fetching live prices for watchlist...

  BTC/USDT     $ 79,795.0000    +1.54%
  ETH/USDT     $  2,346.4000    +0.97%
  SOL/USDT     $     84.0900    +0.21%
  BNB/USDT     $    622.7140    +0.84%
  ADA/USDT     $      0.2498    +0.20%

--- DataFrame View ---
  symbol      price  change_24h  volume_24h  high_24h    low_24h
BTC/USDT 79795.0000        1.54      420.87 80764.600 78219.5000
ETH/USDT  2346.4000        0.97      357.41  2399.180  2309.3100
SOL/USDT    84.0900        0.21       40.91    85.890    83.2700
BNB/USDT   622.7140        0.84       23.91   638.893   615.5750
ADA/USDT     0.2498        0.20        5.09     0.255     0.2471

Best performer (24h):  BTC/USDT (1.54%)
Worst performer (24h): ADA/USDT (0.2%)
Highest volume:        BTC/USDT ($420.87M)


In [16]:

# =============================================================================
# CELL 5 — Fetch OHLCV Candlestick Data (The Foundation of All Trading)
# OHLCV = Open, High, Low, Close, Volume
# This is the raw data your RL agent will be trained on.
# =============================================================================

import ccxt
import pandas as pd
import time

exchange = ccxt.kucoin({'enableRateLimit': True})

def fetch_ohlcv(symbol, timeframe='1d', limit=30):
    """
    Fetch candlestick data for a symbol.

    Args:
        symbol:    Trading pair e.g. 'BTC/USDT'
        timeframe: Candle size — '1m', '5m', '1h', '4h', '1d', '1w'
        limit:     Number of candles to fetch

    Returns:
        Pandas DataFrame with OHLCV columns
    """
    try:
        raw = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
        df = pd.DataFrame(raw, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        df['symbol'] = symbol
        df['timeframe'] = timeframe
        df = df.set_index('timestamp')
        return df
    except Exception as e:
        print(f"Error fetching {symbol}: {e}")
        return None

# Fetch last 30 days of BTC daily candles
btc_daily = fetch_ohlcv('BTC/USDT', timeframe='1d', limit=30)

print("BTC/USDT — Last 30 Days (Daily Candles)")
print("="*70)
print(btc_daily[['open', 'high', 'low', 'close', 'volume']].to_string())

# Quick stats
print("\n--- 30-Day Stats ---")
print(f"Period High:     ${btc_daily['high'].max():,.2f}")
print(f"Period Low:      ${btc_daily['low'].min():,.2f}")
print(f"Period Range:    ${btc_daily['high'].max() - btc_daily['low'].min():,.2f}")
print(f"Avg Daily Vol:   {btc_daily['volume'].mean():,.0f} BTC")
print(f"30d Return:      {((btc_daily['close'].iloc[-1] / btc_daily['close'].iloc[0]) - 1) * 100:.2f}%")

# Add a simple daily return column
btc_daily['daily_return'] = btc_daily['close'].pct_change() * 100
print(f"Best day:        {btc_daily['daily_return'].max():.2f}%")
print(f"Worst day:       {btc_daily['daily_return'].min():.2f}%")
print(f"Positive days:   {(btc_daily['daily_return'] > 0).sum()} / {len(btc_daily)}")


BTC/USDT — Last 30 Days (Daily Candles)
               open     high      low    close       volume
timestamp                                                  
2026-04-05  67302.5  69128.6  66613.4  69018.3   974.294556
2026-04-06  69023.6  70343.5  68317.6  68851.0  1944.720537
2026-04-07  68850.9  72753.8  67731.2  71924.5  2629.087626
2026-04-08  71919.8  72858.1  70709.0  71075.4  1734.829000
2026-04-09  71075.4  73142.9  70472.3  71809.3  1929.131468
2026-04-10  71791.9  73445.8  71430.5  72956.9  1689.973046
2026-04-11  72957.0  73801.1  72519.1  73040.4  1137.176540
2026-04-12  73040.4  73131.9  70505.3  70742.3  1768.259756
2026-04-13  70742.4  74883.2  70571.2  74410.8  2641.729161
2026-04-14  74410.8  76040.9  73810.9  74141.5  2299.032785
2026-04-15  74141.5  75426.7  73520.4  74810.8  1923.001903
2026-04-16  74817.2  75516.0  73311.3  75146.4  2074.173594
2026-04-17  75146.5  78353.3  74541.7  77073.0  2775.975623
2026-04-18  77073.0  77414.9  75444.4  75697.7  1222.324424


In [17]:

# =============================================================================
# CELL 6 — Save Data to CSV (Your Local Data Store for Now)
# Before setting up Supabase on Day 3, save to CSV.
# =============================================================================

import os

# Create a data directory
os.makedirs('trading_data', exist_ok=True)

# Save BTC daily data
btc_daily.to_csv('trading_data/BTC_USDT_1d.csv')
print("Saved: trading_data/BTC_USDT_1d.csv")

# Fetch and save ETH too
eth_daily = fetch_ohlcv('ETH/USDT', timeframe='1d', limit=30)
eth_daily.to_csv('trading_data/ETH_USDT_1d.csv')
print("Saved: trading_data/ETH_USDT_1d.csv")

# Verify by reloading
loaded_btc = pd.read_csv('trading_data/BTC_USDT_1d.csv', index_col='timestamp', parse_dates=True)
print(f"\nReloaded BTC data: {len(loaded_btc)} rows, {loaded_btc.shape[1]} columns")
print(f"Date range: {loaded_btc.index[0].date()} to {loaded_btc.index[-1].date()}")

Saved: trading_data/BTC_USDT_1d.csv
Saved: trading_data/ETH_USDT_1d.csv

Reloaded BTC data: 30 rows, 8 columns
Date range: 2026-04-05 to 2026-05-04


In [21]:
# =============================================================================
# CELL 7 — Alpaca Paper Trading Setup (Updated for alpaca-py)
# Sign up free at: https://alpaca.markets
# Dashboard → API Keys → Generate (make sure you're on PAPER tab)
# =============================================================================

from alpaca.trading.client import TradingClient

ALPACA_API_KEY    = "PKSDCXAUDKSXSH6YKPY4CWN6WY"
ALPACA_SECRET_KEY = "EX54WVvXKrYQS2CCNq62XAo3wATJVo5MkSa4hZo3Tesk"

try:
    api = TradingClient(
        api_key=ALPACA_API_KEY,
        secret_key=ALPACA_SECRET_KEY,
        paper=True   # True = paper trading, never real money
    )

    account = api.get_account()

    print("ALPACA PAPER ACCOUNT")
    print("="*40)
    print(f"Status:          {account.status}")
    print(f"Paper Balance:   ${float(account.cash):,.2f}")
    print(f"Portfolio Value: ${float(account.portfolio_value):,.2f}")
    print(f"Buying Power:    ${float(account.buying_power):,.2f}")
    print(f"Day Trades Left: {account.daytrade_count}")

except Exception as e:
    print(f"Alpaca not configured yet: {e}")
    print("-> Sign up at https://alpaca.markets")
    print("-> Go to Dashboard → Paper Trading → API Keys → Generate New Key")
    print("-> Paste your keys into ALPACA_API_KEY and ALPACA_SECRET_KEY above")



ALPACA PAPER ACCOUNT
Status:          AccountStatus.ACTIVE
Paper Balance:   $100,000.00
Portfolio Value: $100,000.00
Buying Power:    $200,000.00
Day Trades Left: 0


In [22]:
# =============================================================================
# CELL 8 — DAY 2: Fetch US Stock Data via yfinance (No API Key Needed)
# yfinance = free Yahoo Finance wrapper. Works without Alpaca keys.
# =============================================================================

# !pip install yfinance --quiet  # Uncomment if not installed

import yfinance as yf
import pandas as pd

def fetch_stock_data(ticker, period='1mo', interval='1d'):
    """
    Fetch US stock data using Yahoo Finance.
    period:   '1d','5d','1mo','3mo','6mo','1y','2y','5y','10y','ytd','max'
    interval: '1m','2m','5m','15m','30m','60m','90m','1h','1d','5d','1wk','1mo'
    """
    stock = yf.Ticker(ticker)
    df = stock.history(period=period, interval=interval)
    df['ticker'] = ticker
    return df

# Fetch last 1 month of major US stocks
us_stocks = ['AAPL', 'MSFT', 'NVDA', 'TSLA', 'AMZN']

print("US STOCKS — Last 30 Days")
print("="*60)

us_data = {}
for ticker in us_stocks:
    df = fetch_stock_data(ticker, period='1mo')
    us_data[ticker] = df
    latest = df.iloc[-1]
    month_return = ((df['Close'].iloc[-1] / df['Close'].iloc[0]) - 1) * 100
    print(f"{ticker:<6} Close: ${latest['Close']:>8.2f}  |  30d Return: {month_return:>+6.2f}%  |  Vol: {int(latest['Volume']/1e6)}M")

# Fetch NIFTY50 equivalent — use Indian ETF as proxy for now
print("\n--- India Proxy (INDA ETF = iShares MSCI India) ---")
india_etf = fetch_stock_data('INDA', period='1mo')
print(f"INDA  Close: ${india_etf['Close'].iloc[-1]:.2f}  |  30d Return: {((india_etf['Close'].iloc[-1]/india_etf['Close'].iloc[0])-1)*100:.2f}%")

US STOCKS — Last 30 Days
AAPL   Close: $  276.83  |  30d Return:  +6.94%  |  Vol: 45M
MSFT   Close: $  413.62  |  30d Return: +10.93%  |  Vol: 26M
NVDA   Close: $  198.48  |  30d Return: +11.73%  |  Vol: 124M
TSLA   Close: $  392.51  |  30d Return: +11.25%  |  Vol: 48M
AMZN   Close: $  272.05  |  30d Return: +27.85%  |  Vol: 48M

--- India Proxy (INDA ETF = iShares MSCI India) ---
INDA  Close: $48.63  |  30d Return: 2.92%


In [23]:
# =============================================================================
# CELL 9 — Build Your First Market Snapshot Function
# This is the earliest version of the data layer your agent will use.
# =============================================================================

import ccxt
import yfinance as yf
import pandas as pd
import datetime

exchange = ccxt.binance({'enableRateLimit': True})

def get_market_snapshot():
    """
    Unified function that fetches live prices from:
    1. Binance (Crypto)
    2. Yahoo Finance (US Stocks)

    Returns a single DataFrame with all assets.
    """
    snapshot = []
    timestamp = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')

    # --- Crypto via Binance ---
    crypto_symbols = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT']
    for symbol in crypto_symbols:
        try:
            ticker = exchange.fetch_ticker(symbol)
            snapshot.append({
                'asset':      symbol,
                'market':     'CRYPTO',
                'price':      round(ticker['last'], 4),
                'change_24h': round(ticker['percentage'], 2),
                'volume_24h_usd': round(ticker['quoteVolume'], 0),
                'timestamp':  timestamp
            })
        except Exception as e:
            print(f"Crypto fetch error {symbol}: {e}")

    # --- US Stocks via Yahoo Finance ---
    stock_tickers = ['AAPL', 'NVDA', 'TSLA']
    for ticker_symbol in stock_tickers:
        try:
            stock = yf.Ticker(ticker_symbol)
            info = stock.fast_info
            snapshot.append({
                'asset':      ticker_symbol,
                'market':     'US_STOCK',
                'price':      round(info.last_price, 2),
                'change_24h': round(((info.last_price - info.previous_close) / info.previous_close) * 100, 2),
                'volume_24h_usd': round(info.three_month_average_volume * info.last_price, 0),
                'timestamp':  timestamp
            })
        except Exception as e:
            print(f"Stock fetch error {ticker_symbol}: {e}")

    return pd.DataFrame(snapshot)

# Run the snapshot
print("Fetching unified market snapshot...\n")
snapshot_df = get_market_snapshot()
print(snapshot_df.to_string(index=False))

# Save snapshot with timestamp
snapshot_df.to_csv(f'trading_data/snapshot_{datetime.datetime.utcnow().strftime("%Y%m%d_%H%M")}.csv', index=False)
print("\nSnapshot saved to trading_data/")

Fetching unified market snapshot...

Crypto fetch error BTC/USDT: binance GET https://api.binance.com/api/v3/exchangeInfo 451  {
  "code": 0,
  "msg": "Service unavailable from a restricted location according to 'b. Eligibility' in https://www.binance.com/en/terms. Please contact customer service if you believe you received this message in error."
}


/tmp/ipykernel_14394/3377393234.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')


Crypto fetch error ETH/USDT: binance GET https://api.binance.com/api/v3/exchangeInfo 451  {
  "code": 0,
  "msg": "Service unavailable from a restricted location according to 'b. Eligibility' in https://www.binance.com/en/terms. Please contact customer service if you believe you received this message in error."
}
Crypto fetch error SOL/USDT: binance GET https://api.binance.com/api/v3/exchangeInfo 451  {
  "code": 0,
  "msg": "Service unavailable from a restricted location according to 'b. Eligibility' in https://www.binance.com/en/terms. Please contact customer service if you believe you received this message in error."
}
asset   market  price  change_24h  volume_24h_usd               timestamp
 AAPL US_STOCK 276.83       -1.17    1.263900e+10 2026-05-05 00:12:35 UTC
 NVDA US_STOCK 198.48        0.18    3.475102e+10 2026-05-05 00:12:35 UTC
 TSLA US_STOCK 392.51        0.32    2.471953e+10 2026-05-05 00:12:35 UTC

Snapshot saved to trading_data/


/tmp/ipykernel_14394/3377393234.py:65: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  snapshot_df.to_csv(f'trading_data/snapshot_{datetime.datetime.utcnow().strftime("%Y%m%d_%H%M")}.csv', index=False)
